In [1]:
# General data analysis and visualisation
import pandas as pd
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import scipy.stats as stats

# Variable transformation
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import OneHotEncoder

import os

# Serialisation
import pickle

import warnings
warnings.filterwarnings('ignore')

In [2]:
data = pd.read_csv('exercise_bank_marketing_data.csv',sep=';')
data.drop(columns=['Unnamed: 0'],inplace=True)

## data processing and feature engineering

In [3]:
# Impute missing values
# 1. Numeric: Fill missing 'balance' with the median
data['balance'] = data['balance'].fillna(data['balance'].median())

# 2. Categorical: Fill 'education' and 'job' with the mode (most common value)
data['education'] = data['education'].fillna(data['education'].mode()[0])
data['job'] = data['job'].fillna(data['job'].mode()[0])

# 3. Categorical (Logical): Fill 'contact' with 'Unknown'
data['contact'] = data['contact'].fillna('Unknown')

# Verify that all missing values have been handled
print(data[['education', 'balance', 'contact', 'job']].isnull().sum())

# Keep only rows where age is logically sound
data = data[data['age'] <= 100]

#convert target to numeric binary
data['target'] = data['target'].map({'yes': 1, 'no': 0})
data['default'] = data['default'].map({'yes': 1, 'no': 0})
data['housing'] = data['housing'].map({'yes': 1, 'no': 0})
data['loan'] = data['loan'].map({'yes': 1, 'no': 0})
# Drop low-variance columns
data = data.drop(columns=['default'])

# Create  new 'contact_recency' feature a
#inverse of the days (1 / pdays).
#If they were contacted 1 day ago, the score is 1 / 1 = 1.0 (Very high recency).
#If they were contacted 100 days ago, the score is 1 / 100 = 0.01 (Low recency).
#If they were never contacted (-1), set the score to 0.0 (Zero recency).
data['contact_recency'] = np.where(data['pdays'] == -1, 
                                 0,               # If -1, recency is 0
                                 1 / data['pdays']) # Otherwise, 1 divided by days

# drop the original 'pdays' column
data = data.drop(columns=['pdays'])

#drop potential data leakage variables
data = data.drop(columns=['post_campaign_action','duration'])

education    0
balance      0
contact      0
job          0
dtype: int64


###  For categorical variables that have high number of categories, group some smaller categories (less than 5% distribution) together. Then create some new features (ie beginning_of_month, mid_month, end_month) and hot encode

In [4]:
# group entrepreneur and self-employed together (makes logical sense, also have similar target rates)
data['job'] = data['job'].replace(
    ['entrepreneur', 'self-employed'], 
    'self-employed/entrepreneur'
)

data['job'] = data['job'].replace(
    ['student', 'unemployed'], 
    'student/unemployed'
)

data['job'] = data['job'].replace(
    ['housemaid', 'services'], 
    'services/housemaid'
)

data['job'] = data['job'].replace(
    ['unknown'], 
    'management'
)

#clean up admin category spelling
data['job'] = data['job'].replace(
    ['admin.'], 
    'admin'
)

In [5]:
data['poutcome'] = data['poutcome'].replace('other', 'unknown')

In [6]:
bins = [0, 10, 20, 31]
labels = ['month_begin', 'mid_month', 'month_end']

# 2. Apply pd.cut to slice the 'day' column into these bins
data['month_phase'] = pd.cut(data['day'], bins=bins, labels=labels)
data = data.drop(columns=['day'])

In [7]:
#months with highest volumes, while rest of month have lower volumes
peak_months = ['may', 'jun', 'jul', 'aug']

# Create a new binary-style categorical column
data['campaign_season'] = np.where(data['month'].isin(peak_months), 'peak_season', 'off_peak')

data = data.drop(columns=['month'])

In [8]:
data['contact'] = data['contact'].replace(
    ['unknown'], 
    'cellular'
)

In [9]:
data['education'] = data['education'].replace(
    ['unknown'], 
    'secondary'
)

In [10]:
data.head()

,age,job,marital,education,balance,housing,loan,contact,campaign,previous,poutcome,target,contact_recency,month_phase,campaign_season
0,30,student/unemployed,married,primary,1787.0,0,0,cellular,1,0,unknown,0,0.00000,mid_month,off_peak
1,33,services/housemaid,married,secondary,4789.0,1,1,cellular,1,4,failure,0,0.00295,mid_month,peak_season
2,35,management,single,tertiary,1350.0,1,0,cellular,1,1,failure,0,0.00303,mid_month,off_peak
3,30,management,married,tertiary,1476.0,1,1,cellular,4,0,unknown,0,0.00000,month_begin,peak_season
4,59,blue-collar,married,secondary,0.0,1,0,cellular,1,0,unknown,0,0.00000,month_begin,peak_season


## train/test split, One-hot-encoding categorical features, scaling continuous numeric features, applying smote and saving

In [12]:
# ==========================================
# 0. Set Up Your Data and Variables
# ==========================================
target_col = 'target'

X = data.drop(columns=[target_col])
y = data[target_col]

# Identify categorical and numeric columns automatically
# Alternatively, you can define these lists manually: categorical_cols = ['job', 'marital', ...]
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f"Categorical columns: {len(categorical_cols)}")
print(f"Numeric columns: {len(numeric_cols)}")

# ==========================================
# 1. Train-Test Split (Must happen first!)
# ==========================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Train shape before processing: {X_train.shape}")

# ==========================================
# 2. One-Hot Encoding
# ==========================================
encoder = OneHotEncoder(drop='first', sparse=False, handle_unknown='ignore')

# Fit and transform on train, ONLY transform on test
X_train_encoded_array = encoder.fit_transform(X_train[categorical_cols])
X_test_encoded_array = encoder.transform(X_test[categorical_cols])

# Extract new column names and convert back to DataFrames
encoded_col_names = encoder.get_feature_names_out(categorical_cols)
data_train_encoded = pd.DataFrame(X_train_encoded_array, columns=encoded_col_names, index=X_train.index)
data_test_encoded = pd.DataFrame(X_test_encoded_array, columns=encoded_col_names, index=X_test.index)

# Drop original categorical columns and attach the encoded ones
X_train = X_train.drop(columns=categorical_cols).join(data_train_encoded)
X_test = X_test.drop(columns=categorical_cols).join(data_test_encoded)

# ==========================================
# 3. Feature Scaling
# ==========================================
# Distance-based algorithms and SMOTE require numeric variables to be on the same scale
scaler = StandardScaler()

# Fit and transform on train, ONLY transform on test
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

# ==========================================
# 4. Apply SMOTE (ONLY on Training Data!)
# ==========================================
# SMOTE generates synthetic samples for the minority class to balance the training set
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f"Train shape after SMOTE: {X_train_smote.shape}")
print(f"Test shape: {X_test.shape}")

# ==========================================
# 5. Save Data to CSV
# ==========================================
# Create a folder to keep things organized
os.makedirs('processed_data', exist_ok=True)

X_train_smote.to_csv('processed_data/X_train_smote.csv', index=False)
y_train_smote.to_csv('processed_data/y_train_smote.csv', index=False)

X_test.to_csv('processed_data/X_test.csv', index=False)
y_test.to_csv('processed_data/y_test.csv', index=False)

print("Pipeline complete. Processed data successfully saved to the /processed_data/ directory.")

Categorical columns: 7
Numeric columns: 7
Train shape before processing: (3649, 14)
Train shape after SMOTE: (6454, 25)
Test shape: (913, 25)
Pipeline complete. Processed data successfully saved to the /processed_data/ directory.
